# VocalVerse1 — QwenFeat Vocal Scorer (Colab)

Runs the **VocalVerse1 (Qwen2-Audio-7B-Instruct + 4 LoRA adapters)** model from
https://huggingface.co/karl-wang/QwenFeat-Vocal-Score

**4 expert-annotated singing dimensions:**
| # | Dimension | What it measures |
|---|---|---|
| 0 | Timbre (音色) | Voice uniqueness, texture, character |
| 1 | Breath control (气息) | Phrase support, stability |
| 2 | Emotional expression (情感) | Expressiveness, resonance |
| 3 | Vocal technique (技巧) | Mastery of singing skills |

**Runtime requirements:**
- GPU: A100 40GB or L4 24GB recommended (Colab Pro). T4 16GB is too small for the 7B model in fp16.
- Disk: ~20 GB for the HF repo + base model
- Time: ~15 min to download + load on first run

**Sections:**
1. Setup & install
2. Download model weights
3. Score single files (upload from local)
4. Score a PopBuTFy amateur/professional pair (if you mount Drive)
5. Compare to VocalVerse2/SingMOS baselines

## 1. Runtime check & install

Run the cells in order:
1. Check GPU (this cell's code cell)
2. Install `huggingface_hub` and run requirements — **re-run after the download cell** to install `qwenaudio/requirements.txt`
3. Download weights (cell 2)
4. Re-run this install cell if needed, then load the model (cell 3)

In [ ]:
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1024**3
    print(f"GPU            : {props.name}")
    print(f"VRAM           : {vram_gb:.1f} GB")
    if vram_gb < 20:
        print("⚠️  Less than 20 GB VRAM — use bfloat16 loading (set USE_BF16=True below)")
    else:
        print("✅ Sufficient VRAM")
else:
    print("❌ No GPU detected — switch runtime to GPU in Runtime > Change runtime type")

In [ ]:
# Step 1: install huggingface_hub so we can download the repo (and its requirements.txt)
!pip install -q huggingface_hub

# Authenticate with Hugging Face to get higher rate limits and faster downloads.
# Get your token at https://huggingface.co/settings/tokens (read-only token is sufficient).
import os
from huggingface_hub import login

HF_TOKEN = "hf_YOUR_TOKEN_HERE"   # ← paste your token here
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

# Step 2: after the download cell runs, install everything the library needs
# (run this cell again after cell 2 if you skipped straight here)
import pathlib
_req = pathlib.Path("/content/QwenFeat/qwenaudio/requirements.txt")
if _req.exists():
    !pip install -q -r {_req}
else:
    print("requirements.txt not found — run the download cell (cell 2) first, then re-run this cell.")

## 2. Download weights from Hugging Face

The full repo is ~65 GB. We download only the files needed:
- `ckpts/Qwen2-Audio-7B-Instruct/` — base model (~15 GB)
- `ckpts/train_ds_4_score_al/` — 4 score LoRA adapters
- `ckpts/train_ds_4_al/` — adapters for dimension 1
- `ckpts/train_ds_4_feat_score_al/` — adapters for dimensions 2 & 3
- `ckpts/generator-lora-32-16-textonly-simple-v2-int4/` — text LoRA (int4)
- The `qwenaudio/` source directory

In [ ]:
import os
from huggingface_hub import snapshot_download

REPO_ID   = "karl-wang/QwenFeat-Vocal-Score"
LOCAL_DIR = "/content/QwenFeat"   # change to Drive path if using persistent storage

# Patterns to include (everything else skipped — saves ~40 GB)
INCLUDE_PATTERNS = [
    # QwenAudio source code
    "qwenaudio/**",
    # Config files (prompt_set.json etc.)
    "config/**",
    # Base model
    "ckpts/Qwen2-Audio-7B-Instruct/**",
    # Score LoRA adapters (4 dimensions)
    "ckpts/train_ds_4_score_al/denoise/0/score/best_model_epoch/8/**",
    "ckpts/train_ds_4_al/denoise/1/score/best_model_epoch_39/**",
    "ckpts/train_ds_4_al/denoise/1/text/best_model_epoch_39/**",
    "ckpts/train_ds_4_feat_score_al/denoise/2/score/best_model_epoch/25/**",
    "ckpts/train_ds_4_feat_score_al/denoise/3/score/best_model_epoch/5/**",
    # Text LoRA (shared across dim 0,2,3)
    "ckpts/generator-lora-32-16-textonly-simple-v2-int4/best_model_epoch_16/**",
]

EXCLUDE_PATTERNS = [
    # Skip audioscore (MuQ) — not needed here
    "audioscore/**",
    # Skip unused LoRA variants
    "ckpts/generator-lora-128*/**",
    "ckpts/generator-lora-16*/**",
    "ckpts/outputs-only/**",
    "ckpts/score_lora*/**",
    "ckpts/SongEvalGenerator/**",
    "ckpts/hubert_pretrain/**",
    "ckpts/whisper_pretrain/**",
]

print(f"Downloading to {LOCAL_DIR} ...")
print("This will take ~10-20 minutes on first run.")

local_path = snapshot_download(
    repo_id=REPO_ID,
    repo_type="model",
    local_dir=LOCAL_DIR,
    allow_patterns=INCLUDE_PATTERNS,
    ignore_patterns=EXCLUDE_PATTERNS,
)
print(f"\nDownloaded to: {local_path}")

# Verify key paths exist
import pathlib
checks = [
    "ckpts/Qwen2-Audio-7B-Instruct",
    "ckpts/train_ds_4_score_al",
    "ckpts/generator-lora-32-16-textonly-simple-v2-int4",
    "qwenaudio/qwenaudio",
    "config/prompt_set.json",
]
for c in checks:
    p = pathlib.Path(LOCAL_DIR) / c
    status = "✅" if p.exists() else "❌ MISSING"
    print(f"  {status}  {c}")

## 3. Patch checkpoint paths & load model

The `infer_service.py` hardcodes relative `ckpts/` paths. We patch them here to use absolute paths inside `LOCAL_DIR`.

In [ ]:
import sys, os, pathlib

LOCAL_DIR = "/content/QwenFeat"   # must match cell above
USE_BF16  = True   # set False if you have >= 24 GB VRAM and want fp16

# Add qwenaudio source to path
sys.path.insert(0, str(pathlib.Path(LOCAL_DIR) / "qwenaudio" / "src"))

# Change working directory so relative ckpt paths in the library resolve correctly
os.chdir(LOCAL_DIR)

# Ensure config/prompt_set.json exists — may be missing if the download cell ran
# without the "config/**" include pattern (fetch it individually if needed).
_config_file = pathlib.Path(LOCAL_DIR) / "config" / "prompt_set.json"
if not _config_file.exists():
    print("config/prompt_set.json missing — downloading now ...")
    from huggingface_hub import hf_hub_download
    hf_hub_download(
        repo_id="karl-wang/QwenFeat-Vocal-Score",
        repo_type="model",
        filename="config/prompt_set.json",
        local_dir=LOCAL_DIR,
    )
    print(f"  ✅ saved to {_config_file}")
else:
    print(f"✅ config/prompt_set.json found")

import qwenaudio.processor
import qwenaudio.prompts

# Monkey-patch the base model path if the library hardcodes .cache
# (Some versions auto-detect; if loading fails, set this env var)
BASE_MODEL_PATH = str(pathlib.Path(LOCAL_DIR) / "ckpts" / "Qwen2-Audio-7B-Instruct")
os.environ["QWEN_AUDIO_BASE"] = BASE_MODEL_PATH

print("Initialising processor (loads base model + 4 LoRA adapters) ...")
print("First load takes 3-5 minutes.")

processor = qwenaudio.processor.ProcessorGroup()

TEXT_LORA = "ckpts/generator-lora-32-16-textonly-simple-v2-int4/best_model_epoch_16/lora_weights"

# Dimension 0 — Timbre
processor.add("ckpts/train_ds_4_score_al/denoise/0/score/best_model_epoch/8", TEXT_LORA)
# Dimension 1 — Breath control (has its own text LoRA)
processor.add(
    "ckpts/train_ds_4_al/denoise/1/score/best_model_epoch_39/lora_weights",
    "ckpts/train_ds_4_al/denoise/1/text/best_model_epoch_39/lora_weights",
)
# Dimension 2 — Emotional expression
processor.add("ckpts/train_ds_4_feat_score_al/denoise/2/score/best_model_epoch/25", TEXT_LORA)
# Dimension 3 — Vocal technique
processor.add("ckpts/train_ds_4_feat_score_al/denoise/3/score/best_model_epoch/5", TEXT_LORA)

processor.models[0].top2_mode = False
processor.models[1].top2_mode = True    # dim 1 uses top2 (two-pass scoring)
processor.models[2].top2_mode = False
processor.models[3].top2_mode = False

print("\n✅ All 4 models loaded")

DIMS = {
    0: "Timbre",
    1: "Breath Control",
    2: "Emotional Expression",
    3: "Vocal Technique",
}

## 4. Inference helpers

In [ ]:
import librosa, warnings
import numpy as np

SR_QWEN = processor.processor.feature_extractor.sampling_rate
MAX_SAMPLES = SR_QWEN * 30   # model supports up to 30 s

def load_audio(path):
    y, _ = librosa.load(str(path), sr=SR_QWEN, mono=True)
    if len(y) > MAX_SAMPLES:
        warnings.warn(f"Clip longer than 30s — truncating: {path}")
        y = y[:MAX_SAMPLES]
    return y

def score_audio(audio_data, dims=(0, 1, 2, 3)):
    """Score audio array across requested dimensions.
    Returns dict: {dim_name: {score, text}}
    """
    results = {}
    for i in dims:
        out = processor.models[i].generate(audio_data, i, simple_model=True)
        results[DIMS[i]] = {
            "score": round(float(out["score"]), 3),
            "text":  out.get("text", ""),
        }
    results["sum_score"] = round(sum(v["score"] for v in results.values() if isinstance(v, dict)), 3)
    return results

def print_scores(label, results):
    print(f"\n── {label} ──")
    for dim, val in results.items():
        if dim == "sum_score":
            print(f"  {'SUM':25} {val:.3f}")
        else:
            print(f"  {dim:25} {val['score']:.3f}")
            if val['text']:
                print(f"    {val['text'][:120]}")

print(f"Sample rate    : {SR_QWEN} Hz")
print(f"Max duration   : 30 s")
print("Helper functions ready.")

## 5. Score uploaded files

Upload WAV/MP3 files from your machine, then run this cell.

In [ ]:
from google.colab import files as colab_files

uploaded = colab_files.upload()   # opens file picker

for filename, content in uploaded.items():
    tmp_path = f"/tmp/{filename}"
    with open(tmp_path, "wb") as f:
        f.write(content)
    print(f"\nScoring: {filename}")
    y = load_audio(tmp_path)
    print(f"  Duration: {len(y)/SR_QWEN:.1f}s")
    results = score_audio(y)
    print_scores(filename, results)

## 6. PopBuTFy amateur vs professional comparison

Mount Google Drive if you have PopBuTFy there, or upload clips directly.
Pairs clips by index `_N.mp3` in amateur and professional folders.

In [ ]:
import re, pathlib, json, warnings, logging
import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*sampling_rate.*")

# WhisperFeatureExtractor prints via the transformers logger, not warnings module
logging.getLogger("transformers.models.whisper.feature_extraction_whisper").setLevel(logging.ERROR)

POPBUTFY_ROOT = "/content/popbutfy"   # ← change to your path

SINGER = "Female1"
SONG   = "my_heart_will_go_on"

def find_folder(root, singer, song, level):
    pat = re.compile(
        rf"^{re.escape(singer)}#singing#{re.escape(song)}_{level}$",
        re.IGNORECASE,
    )
    for d in sorted(pathlib.Path(root).iterdir()):
        if d.is_dir() and pat.match(d.name):
            return d
    return None

def sorted_clips(folder):
    return sorted(
        folder.glob("*.mp3"),
        key=lambda p: int(re.search(r"_(\d+)\.mp3$", p.name).group(1)),
    )

am_dir  = find_folder(POPBUTFY_ROOT, SINGER, SONG, "Amateur")
pro_dir = find_folder(POPBUTFY_ROOT, SINGER, SONG, "Professional")

if am_dir is None or pro_dir is None:
    print(f"Could not find folders. Check POPBUTFY_ROOT and SINGER/SONG values.")
else:
    am_clips  = sorted_clips(am_dir)
    pro_clips = sorted_clips(pro_dir)
    n = min(len(am_clips), len(pro_clips))
    print(f"Song: {SONG}  |  Singer: {SINGER}  |  Clip pairs: {n}")
    print(f"\n{'Clip':6} | {'Timbre':8} {'Breath':8} {'Emotion':8} {'Technique':10} {'Sum':8} | Source")
    print("─" * 72)

    am_agg  = {d: [] for d in DIMS.values()}
    pro_agg = {d: [] for d in DIMS.values()}

    for i in range(n):
        for clips, agg, label in [
            (am_clips,  am_agg,  "Amateur"),
            (pro_clips, pro_agg, "Pro"),
        ]:
            y = load_audio(clips[i])
            res = score_audio(y)
            for dim in DIMS.values():
                agg[dim].append(res[dim]["score"])
            scores_str = "  ".join(f"{res[d]['score']:.2f}" for d in DIMS.values())
            print(f"  {i:<4}  | {scores_str}  {res['sum_score']:7.2f}   | {label}")

    print("─" * 72)
    for label, agg in [("Amateur mean", am_agg), ("Pro mean", pro_agg)]:
        means = "  ".join(f"{np.mean(agg[d]):.2f}" for d in DIMS.values())
        total = sum(np.mean(agg[d]) for d in DIMS.values())
        print(f"  {label:12}| {means}  {total:7.2f}")

## 6b. Build population baselines — ALL PopBuTFy clips

Scans every `_Amateur` / `_Professional` folder, scores every clip,
computes mean/median/p25/p75/std per level, and saves JSON to `/content/vocalverse1_baselines.json`.

Tune `BATCH_SIZE` (default 4) and `NUM_WORKERS` (default 4) at the top of the cell.


In [ ]:
## 6b. Build population baselines — ALL PopBuTFy clips
#
# FAST_MODE = True  (default) — uses dim 0 (Timbre) LoRA only → 1 forward pass/clip
#                               ~4× faster, outputs a single aggregate quality score
# FAST_MODE = False           — all 4 dims, ~30s/clip, full dimension breakdown
#
# Architecture notes:
#   • generate() is single-sample only — no true GPU batching possible without
#     patching qwenaudio (QwenAudioScoreModel accepts batched tensors internally
#     but the generate() wrapper takes one audio array per call)
#   • top2_mode on dim 1 (Breath): if top prediction is "3" with <90% confidence,
#     uses the 2nd-most-probable score instead — avoids collapse to midpoint

import re, pathlib, json, warnings, logging, time
import numpy as np
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*sampling_rate.*")
logging.getLogger("transformers.models.whisper.feature_extraction_whisper").setLevel(logging.ERROR)

# ── config ────────────────────────────────────────────────────────────────────
POPBUTFY_ROOT    = "/content/popbutfy"   # ← change to your path
OUTPUT_JSON      = "/content/vocalverse1_baselines.json"
PREFETCH_WORKERS = 4                     # CPU threads for audio loading
FAST_MODE        = True                  # True = 1 pass/clip (aggregate); False = 4 passes/clip (all dims)
# ─────────────────────────────────────────────────────────────────────────────

ACTIVE_DIMS = [0] if FAST_MODE else [0, 1, 2, 3]
mode_label  = "FAST (dim 0 / aggregate)" if FAST_MODE else "FULL (4 dimensions)"

audio_exts = {".wav", ".flac", ".mp3"}
splits = {"amateur": [], "professional": []}

for folder in sorted(pathlib.Path(POPBUTFY_ROOT).iterdir()):
    if not folder.is_dir():
        continue
    name_lower = folder.name.lower()
    if name_lower.endswith("_amateur"):
        label = "amateur"
    elif name_lower.endswith("_professional"):
        label = "professional"
    else:
        continue
    for f in sorted(folder.iterdir()):
        if f.suffix.lower() in audio_exts:
            splits[label].append(f)

total_clips   = len(splits["amateur"]) + len(splits["professional"])
secs_per_clip = 8 if FAST_MODE else 30
est_hours     = total_clips * secs_per_clip / 3600
print(f"Mode           : {mode_label}")
print(f"Discovered     : amateur={len(splits['amateur'])}, professional={len(splits['professional'])} (total={total_clips})")
print(f"Est. time      : {est_hours:.1f}h @ ~{secs_per_clip}s/clip (recalculated after first clip)")
print(f"Prefetch workers: {PREFETCH_WORKERS}")
print()

def percentile_stats(values):
    arr = np.array([v for v in values if np.isfinite(v)], dtype=np.float64)
    if len(arr) == 0:
        return {"mean": None, "median": None, "p25": None, "p75": None, "std": None, "n": 0}
    return {
        "mean":   round(float(np.mean(arr)), 4),
        "median": round(float(np.median(arr)), 4),
        "p25":    round(float(np.percentile(arr, 25)), 4),
        "p75":    round(float(np.percentile(arr, 75)), 4),
        "std":    round(float(np.std(arr)), 4),
        "n":      int(len(arr)),
    }

def score_one(y):
    """Score a single audio array across ACTIVE_DIMS."""
    res = {}
    for i in ACTIVE_DIMS:
        out = processor.models[i].generate(y, i, simple_model=True)
        res[DIMS[i]] = {"score": round(float(out["score"]), 3), "text": out.get("text", "")}
    res["sum_score"] = round(sum(v["score"] for v in res.values() if isinstance(v, dict)), 3)
    return res

ACTIVE_DIM_NAMES = [DIMS[i] for i in ACTIVE_DIMS]
scores = {label: {d: [] for d in ACTIVE_DIM_NAMES + ["sum"]} for label in splits}
errors = {label: 0 for label in splits}
t0_global = time.time()

for label, files in splits.items():
    print(f"Loading {label} audio ({len(files)} clips) in parallel ...")
    loaded = [None] * len(files)

    with ThreadPoolExecutor(max_workers=PREFETCH_WORKERS) as pool:
        future_to_idx = {pool.submit(load_audio, f): i for i, f in enumerate(files)}
        for fut in tqdm(as_completed(future_to_idx), total=len(files), desc=f"  load {label}"):
            idx = future_to_idx[fut]
            try:
                loaded[idx] = fut.result()
            except Exception as e:
                warnings.warn(f"[load error] {files[idx].name}: {e}")
                errors[label] += 1

    valid = [(i, y) for i, y in enumerate(loaded) if y is not None]
    passes = len(ACTIVE_DIMS)
    print(f"  Scoring {len(valid)} clips ({passes} forward pass{'es' if passes > 1 else ''} each) ...")

    t0 = time.time()
    for n_done, (_, y) in enumerate(tqdm(valid, desc=f"  score {label}")):
        try:
            res = score_one(y)
            for d in ACTIVE_DIM_NAMES:
                scores[label][d].append(res[d]["score"])
            scores[label]["sum"].append(res["sum_score"])
            if n_done == 0:
                elapsed = time.time() - t0
                remaining = elapsed * (len(valid) - 1)
                print(f"    First clip: {elapsed:.1f}s  →  est. remaining for {label}: {remaining/3600:.2f}h")
        except Exception as e:
            warnings.warn(f"[score error] clip {n_done}: {e}")
            errors[label] += 1

print(f"\nDone in {(time.time()-t0_global)/3600:.2f}h. "
      f"Errors: amateur={errors['amateur']}, professional={errors['professional']}")

stats = {}
for label in splits:
    stats[label] = {d: percentile_stats(scores[label][d]) for d in ACTIVE_DIM_NAMES + ["sum"]}

print(f"\n{'Dimension':<24} {'Amateur mean':>14} {'Pro mean':>10} {'Sep':>8}")
print("─" * 60)
for d in ACTIVE_DIM_NAMES + ["sum"]:
    am = stats["amateur"][d]["mean"]
    pr = stats["professional"][d]["mean"]
    sep = (pr - am) if (am is not None and pr is not None) else None
    sep_str = f"{sep:+.3f}" if sep is not None else "n/a"
    print(f"  {d:<22} {str(round(am,3)) if am else 'n/a':>14} "
          f"{str(round(pr,3)) if pr else 'n/a':>10} {sep_str:>8}")

output = {
    "amateur":      {d: stats["amateur"][d] for d in ACTIVE_DIM_NAMES + ["sum"]},
    "professional": {d: stats["professional"][d] for d in ACTIVE_DIM_NAMES + ["sum"]},
    "_meta": {
        "model":            "VocalVerse1 / Qwen2-Audio-7B + 4 LoRA",
        "mode":             mode_label,
        "active_dims":      ACTIVE_DIM_NAMES,
        "n_amateur":        len(splits["amateur"]),
        "n_professional":   len(splits["professional"]),
        "total_time_hours": round((time.time() - t0_global) / 3600, 3),
    },
}
with open(OUTPUT_JSON, "w") as fh:
    json.dump(output, fh, indent=2)
print(f"\nBaselines saved → {OUTPUT_JSON}")

In [ ]:
# Download the baselines JSON to your local machine
from google.colab import files as colab_files
colab_files.download(OUTPUT_JSON)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def radar_chart(am_means, pro_means, dims, title="VocalVerse1 — Amateur vs Professional"):
    N = len(dims)
    angles = [n / N * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    am_vals  = [am_means[d] for d in dims] + [am_means[dims[0]]]
    pro_vals = [pro_means[d] for d in dims] + [pro_means[dims[0]]]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    ax.plot(angles, am_vals,  "o-", linewidth=2, color="#e74c3c", label="Amateur")
    ax.fill(angles, am_vals,  alpha=0.15, color="#e74c3c")
    ax.plot(angles, pro_vals, "o-", linewidth=2, color="#2ecc71", label="Professional")
    ax.fill(angles, pro_vals, alpha=0.15, color="#2ecc71")

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(dims, fontsize=12)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8])
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=11)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.show()

# Only run if am_agg / pro_agg exist from cell above
try:
    am_means  = {d: float(np.mean(am_agg[d]))  for d in DIMS.values()}
    pro_means = {d: float(np.mean(pro_agg[d])) for d in DIMS.values()}
    radar_chart(am_means, pro_means, list(DIMS.values()))

    print("\nDimension separations (Pro − Amateur):")
    for d in DIMS.values():
        sep = pro_means[d] - am_means[d]
        bar = "▓" * int(abs(sep) * 40)
        direction = "+" if sep >= 0 else "-"
        print(f"  {d:25} {direction}{abs(sep):.3f}  {bar}")
except NameError:
    print("Run the PopBuTFy comparison cell first (cell 6).")

## 8. Save scores to CSV

Saves per-clip scores to `/content/vocalverse1_scores.csv` and downloads it.

In [ ]:
import csv, io

CSV_PATH = "/content/vocalverse1_scores.csv"

try:
    rows = []
    for i in range(n):
        for clips, level in [(am_clips, "amateur"), (pro_clips, "professional")]:
            y = load_audio(clips[i])
            res = score_audio(y)
            row = {
                "clip_index": i,
                "level":      level,
                "file":       clips[i].name,
            }
            for dim in DIMS.values():
                row[dim.lower().replace(" ", "_")] = res[dim]["score"]
            row["sum_score"] = res["sum_score"]
            rows.append(row)

    if rows:
        with open(CSV_PATH, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
        print(f"Saved {len(rows)} rows → {CSV_PATH}")
        from google.colab import files as colab_files
        colab_files.download(CSV_PATH)
    else:
        print("No rows to save — run cells 6 first.")
except NameError:
    print("Run the PopBuTFy comparison cell first (cell 6).")

## 9. Troubleshooting

**OOM on load:** Set `USE_BF16=True` in cell 3 and restart. If still OOM, use Colab Pro with A100.

**`ModuleNotFoundError: qwenaudio`:** The `sys.path.insert` in cell 3 must run before any import. Re-run cells 3 → 4 in order.

**Base model not found:** The library may look in `~/.cache/huggingface`. If so:
```python
import os
os.environ["TRANSFORMERS_CACHE"] = "/content/QwenFeat/ckpts"
os.environ["HF_HOME"] = "/content/QwenFeat/ckpts"
```
Then restart and re-run.

**LoRA path not found:** Navigate into `LOCAL_DIR/ckpts/` and verify the exact subdirectory names match the paths in cell 3. The HF snapshot may have slightly different nesting.

**Slow download:** Add `hf_transfer` for faster HF downloads:
```bash
pip install hf_transfer
HF_HUB_ENABLE_HF_TRANSFER=1 python ...
```